# 05 · Air quality — next-day NO₂ and PM2.5 forecasts for all 60 CRISP sites

**CRISP module:** Air quality / Industrial · **Task:** forecast tomorrow's daily-mean NO₂ and PM2.5 at every
CRISP site · **Model:** one LightGBM model shared by all 60 sites

Data: CAMS global atmospheric-composition fields (which assimilate satellite retrievals, including Sentinel-5P) and
ERA5 weather, both via the free Open-Meteo APIs, Aug 2022 → Sep 2026. Chronological split — train until Mar 2025,
validate Apr–Sep 2025, **test on the whole following year (Oct 2025 – Sep 2026)**, so every season is tested.
Scored against persistence baselines ("tomorrow = today"), which are hard to beat for day-ahead air quality.
CPU only, about 10 min (most of it downloading).

In [ ]:
# ── Setup: run once per Colab session (Runtime ▸ Change runtime type ▸ T4 GPU) ──
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/inayatarshad/trainedmodels_gstp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    DRIVE = Path("/content/drive/MyDrive/gstp_runs")       # checkpoints + results persist here
    REPO = WORK / "trainedmodels_gstp"
    if not REPO.exists():
        if subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)]).returncode != 0:
            # Private repo? Upload the repo zip to MyDrive/trainedmodels_gstp.zip and re-run.
            shutil.unpack_archive("/content/drive/MyDrive/trainedmodels_gstp.zip", WORK)
            REPO = next(WORK.glob("trainedmodels_gstp*"))
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:  # local / Kaggle
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO)
    WORK = DRIVE = REPO / "runs"

sys.path.insert(0, str(REPO / "src"))
DRIVE.mkdir(parents=True, exist_ok=True)

import torch
from gstp.utils import seed_everything, get_device
seed_everything(42)
print("Repo:", REPO, "| device:", get_device(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
MODULE = "air_quality"
from gstp import air_quality as AQ
from gstp.metrics import regression_scores
from gstp.utils import load_sites, save_json
import numpy as np, pandas as pd, matplotlib.pyplot as plt

RESULTS = REPO / "results" / MODULE
CKPT_DIR = DRIVE / MODULE
CACHE = DRIVE / "aq_cache"
for p in (RESULTS / "figures", CKPT_DIR, CACHE):
    p.mkdir(parents=True, exist_ok=True)
START, END = "2022-08-01", "2026-09-30"
VAL_START, TEST_START = pd.Timestamp("2025-04-01"), pd.Timestamp("2025-10-01")
sites = load_sites()
print(len(sites), "CRISP sites")

In [ ]:
hourly = AQ.fetch_all(sites, START, END, CACHE)
daily = AQ.to_daily(hourly)
daily.to_csv(RESULTS / "daily_site_data.csv.gz", index=False)
print(daily.shape, daily["date"].min().date(), "→", daily["date"].max().date())

In [ ]:
results, models = {}, {}
for target in ("no2", "pm25"):
    for weather in (True, False):
        tag = f"{target}_{'with' if weather else 'no'}_nextday_weather"
        df = AQ.make_features(daily, target, use_next_day_weather=weather)
        tr, va, te = AQ.chrono_split(df, VAL_START, TEST_START)
        model, cols = AQ.train_lgbm(tr, va)
        pred = AQ.predict_lgbm(model, te, cols)
        res = {"LightGBM": regression_scores(te["target"], pred)}
        for name, b in AQ.baselines(te, AQ.TARGETS[target]).items():
            ok = ~np.isnan(b)
            res[name] = regression_scores(te["target"].values[ok], b[ok])
        res["skill_vs_persistence_mae"] = 1 - res["LightGBM"]["mae"] / res["persistence"]["mae"]
        res["n_train"], res["n_val"], res["n_test"] = len(tr), len(va), len(te)
        results[tag], models[tag] = res, (model, cols, te, pred)
        model.save_model(str(CKPT_DIR / f"lgbm_{tag}.txt"))
        print(f"{tag:<34} MAE {res['LightGBM']['mae']:.2f} vs persistence {res['persistence']['mae']:.2f}"
              f"  → skill {res['skill_vs_persistence_mae']:+.1%}   R² {res['LightGBM']['r2']:.3f}")
save_json(results, RESULTS / "test_metrics.json")

In [ ]:
rows = []
for tag, r in results.items():
    for m in ("LightGBM", "persistence", "weekly_persistence", "mean_7day"):
        rows.append({"setting": tag, "model": m, **{k: r[m][k] for k in ("mae", "rmse", "r2", "bias")}})
table = pd.DataFrame(rows)
table.to_csv(RESULTS / "results_table.csv", index=False)
table.style.format(precision=3).hide(axis="index")

## Per-site accuracy, feature importance and example forecasts

In [ ]:
model, cols, te, pred = models["no2_with_nextday_weather"]
te = te.assign(pred=pred, persist=te["nitrogen_dioxide_lag0"])
per_site = te.groupby("site_id", observed=True).apply(
    lambda g: pd.Series({"MAE_model": np.mean(np.abs(g.pred - g.target)),
                         "MAE_persistence": np.mean(np.abs(g.persist - g.target)),
                         "mean_NO2": g.target.mean()})).sort_values("mean_NO2", ascending=False)
per_site["skill"] = 1 - per_site.MAE_model / per_site.MAE_persistence
per_site.to_csv(RESULTS / "per_site_no2.csv")
print(f"Model beats persistence at {(per_site.skill > 0).sum()} of {len(per_site)} sites")

imp = pd.Series(model.feature_importance("gain"), index=cols).sort_values(ascending=False).head(15)
ax = imp[::-1].plot.barh(figsize=(6, 4.5), title="NO₂ model — top features (gain)")
ax.figure.tight_layout(); ax.figure.savefig(RESULTS / "figures" / "feature_importance_no2.png", dpi=150)

fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
for ax, sid in zip(axes, ["delhi-ncr", "madrid-sur", "manchester-centre", "ruhr-essen"]):
    g = te[te.site_id == sid].sort_values("target_date")
    ax.plot(g.target_date, g.target, label="observed (CAMS)", lw=1)
    ax.plot(g.target_date, g.pred, label="LightGBM forecast", lw=1)
    ax.set_title(sid, fontsize=9); ax.set_ylabel("NO₂ µg/m³")
axes[0].legend(fontsize=8); fig.tight_layout(); fig.savefig(RESULTS / "figures" / "no2_forecasts.png", dpi=150)

In [ ]:
save_json({"results": results, "period": [START, END], "val_start": str(VAL_START.date()),
           "test_start": str(TEST_START.date()), "data": "CAMS global via Open-Meteo; ERA5 via Open-Meteo archive"},
          RESULTS / "summary.json")

In [ ]:
# ── Package results: metrics + figures go to Drive as a zip you can download ──
import shutil
archive = shutil.make_archive(str(DRIVE / f"results_{MODULE}"), "zip", RESULTS)
print("Results zip :", archive)
print("Checkpoints :", sorted(str(p) for p in CKPT_DIR.glob("*.pt")))